# Creating a playing cards dataset
This notebook is a guide through the creation of a dataset of playing cards. The cards are labeled with their name (ex: "2c" for "2 of spades", "Kh" for King for hearts) and with the bounding boxes delimiting their printed corners.
> _Why bounding boxes around the corners, and not around the whole card ?_<br>Because in real conditions, more often than not, cards are partially covered. And the corner of a card is the minimum information you need to identify it.


# Prerequisites 

### A. In addition to opencv and numpy, you need the following python packages :
1. **imgaug** : https://github.com/aleju/imgaug 
> Helps with image augmentation
2. **shapely** : https://github.com/Toblerity/Shapely
> For the manipulation and analysis of geometric objects in the Cartesian plane. It is useful here when we want to check if the bounding box of a card corner is covered by another card
3. **tqdm** : https://github.com/tqdm/tqdm
> A progress bar tool. Not mandatory but convenient when you generate thousands of images

### B. Get the Describable Textures Dataset (DTD)
> A collection of textural images in the wild (https://www.robots.ox.ac.uk/~vgg/data/dtd/). It is probably not its original goal, but it is used here as an easy way to generate various backgrounds for our images.


## Imports

In [ ]:
import numpy as np
import cv2
import os
from tqdm import tqdm
import random
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as patches
import pickle
from glob import glob 
import imgaug as ia
from imgaug import augmenters as iaa
from shapely.geometry import Polygon


In [ ]:
cardW=238
cardH=334
cornerXmin=2.4
cornerXmax=35.2
cornerYmin=6.8
cornerYmax=88

# We convert the measures from mm to pixels: multiply by an arbitrary factor 'zoom'
# You shouldn't need to change this
zoom=1
cardW*=zoom
cardH*=zoom
cornerXmin=int(cornerXmin*zoom)
cornerXmax=int(cornerXmax*zoom)
cornerYmin=int(cornerYmin*zoom)
cornerYmax=int(cornerYmax*zoom)

## Some convenient functions used in this notebook

In [ ]:

def display_img(img,polygons=[],channels="bgr",size=9):
    """
        Function to display an inline image, and draw optional polygons (bounding boxes, convex hulls) on it.
        Use the param 'channels' to specify the order of the channels ("bgr" for an image coming from OpenCV world)
    """
    if not isinstance(polygons,list):
        polygons=[polygons]    
    if channels=="bgr": # bgr (cv2 image)
        nb_channels=img.shape[2]
        if nb_channels==4:
            img=cv2.cvtColor(img,cv2.COLOR_BGRA2RGBA)
        else:
            img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)    
    fig,ax=plt.subplots(figsize=(size,size))
    ax.set_facecolor((0,0,0))
    ax.imshow(img)
    for polygon in polygons:
        # An polygon has either shape (n,2), 
        # either (n,1,2) if it is a cv2 contour (like convex hull).
        # In the latter case, reshape in (n,2)
        if len(polygon.shape)==3:
            polygon=polygon.reshape(-1,2)
        patch=patches.Polygon(polygon,linewidth=1,edgecolor='g',facecolor='none')
        ax.add_patch(patch)

def give_me_filename(dirname, suffixes, prefix=""):
    """
        Function that returns a filename or a list of filenames in directory 'dirname'
        that does not exist yet. If 'suffixes' is a list, one filename per suffix in 'suffixes':
        filename = dirname + "/" + prefix + random number + "." + suffix
        Same random number for all the file name
        Ex: 
        > give_me_filename("dir","jpg", prefix="prefix")
        'dir/prefix408290659.jpg'
        > give_me_filename("dir",["jpg","xml"])
        ['dir/877739594.jpg', 'dir/877739594.xml']        
    """
    if not isinstance(suffixes, list):
        suffixes=[suffixes]
    
    suffixes=[p if p[0]=='.' else '.'+p for p in suffixes]
          
    while True:
        bname="%09d"%random.randint(0,999999999)
        fnames=[]
        for suffix in suffixes:
            fname=os.path.join(dirname,prefix+bname+suffix)
            if not os.path.isfile(fname):
                fnames.append(fname)
                
        if len(fnames) == len(suffixes): break
    
    if len(fnames)==1:
        return fnames[0]
    else:
        return fnames

# Define global variables

In [ ]:
data_dir="data" # Directory that will contain all kinds of data (the data we download and the data we generate)

if not os.path.isdir(data_dir):
    os.makedirs(data_dir)

card_suits=['S','H','D','C']
card_values=['A','K','Q','J','10','9','8','7','6','5','4','3','2']

# Pickle file containing the background images from the DTD
backgrounds_pck_fn=data_dir+"/backgrounds.pck"

# Pickle file containing the card images
cards_pck_fn=data_dir+"/cards.pck"


# imgW,imgH: dimensions of the generated dataset images 
imgW=1280
imgH=1280


refCard=np.array([[0,0],[cardW,0],[cardW,cardH],[0,cardH]],dtype=np.float32)
refCardRot=np.array([[cardW,0],[cardW,cardH],[0,cardH],[0,0]],dtype=np.float32)
refCornerHL=np.array([[cornerXmin,cornerYmin],[cornerXmax,cornerYmin],[cornerXmax,cornerYmax],[cornerXmin,cornerYmax]],dtype=np.float32)
refCornerLR=np.array([[cardW-cornerXmax,cardH-cornerYmax],[cardW-cornerXmin,cardH-cornerYmax],[cardW-cornerXmin,cardH-cornerYmin],[cardW-cornerXmax,cardH-cornerYmin]],dtype=np.float32)
refCorners=np.array([refCornerHL,refCornerLR])


# Get Describable Textures Dataset (DTD)
A convenient way to generate backgrounds for the images of the cards dataset

### Download DTD (1x)

In [ ]:
#!wget https://www.robots.ox.ac.uk/~vgg/data/dtd/download/dtd-r1.0.1.tar.gz

### Extract the DTD (1x)

In [ ]:
#!tar xf dtd-r1.0.1.tar.gz

### Load all *jpg from dtd subdirectories and save them in a pickle file (1x)

The next times, we will directly load the pickle file 

In [ ]:
'''
dtd_dir="dtd/images/"
bg_images=[]
for subdir in glob(dtd_dir+"/*"):
    for f in glob(subdir+"/*.jpg"):
        bg_images.append(mpimg.imread(f))
print("Nb of images loaded :",len(bg_images))
print("Saved in :",backgrounds_pck_fn)
pickle.dump(bg_images,open(backgrounds_pck_fn,'wb'))
'''

In [ ]:
# Clean-up
#!rm -r dtd
#!rm dtd-r1.0.1.tar.gz

### Load the backgounds pickle file in 'backgrounds'
'backgrounds' is an instance of the class Backgrounds
To get a random background image, call the method : backgrounds.get_random

In [ ]:
class Backgrounds():
    def __init__(self,backgrounds_pck_fn=backgrounds_pck_fn):
        self._images=pickle.load(open(backgrounds_pck_fn,'rb'))
        self._nb_images=len(self._images)
        print("Nb of images loaded :", self._nb_images)
    def get_random(self, display=False):
        bg=self._images[random.randint(0,self._nb_images-1)]
        if display: plt.imshow(bg)
        return bg
    
backgrounds = Backgrounds()


In [ ]:
# Test: display a random background
_=backgrounds.get_random(display=True)

# Extraction of the cards from pictures or video 

### Define the alphamask
The alphamask has 2 purposes:
- clean the border of the detected cards,
- make that border transparent. Cards are not perfect rectangles because corners are rounded. We need to make transparent the zone between the real card and its bounding rectangle, otherwise this zone will be visible in the final generated images of the dataset


In [ ]:
bord_size=3 # bord_size alpha=0
alphamask=np.ones((cardH,cardW),dtype=np.uint8)*255
cv2.rectangle(alphamask,(0,0),(cardW-1,cardH-1),0,bord_size)
cv2.line(alphamask,(bord_size*3,0),(0,bord_size*3),0,bord_size)
cv2.line(alphamask,(cardW-bord_size*3,0),(cardW,bord_size*3),0,bord_size)
cv2.line(alphamask,(0,cardH-bord_size*3),(bord_size*3,cardH),0,bord_size)
cv2.line(alphamask,(cardW-bord_size*3,cardH),(cardW,cardH-bord_size*3),0,bord_size)
plt.figure(figsize=(10,10))
plt.imshow(alphamask)

In [ ]:
#load the template dataset
TEMPLATE_DATASET_PATH = "./templates"
imgs_dir=TEMPLATE_DATASET_PATH
imgs_fns=glob(imgs_dir+"/*.png")

### Before going on, check that everything looks good
We randomly choose and display one of the extracted card. We also draw on the card the 2 polygons defined by refCornerHL and refCornerLR. Check that the value and suit symbols are well inside the polygons. If not, check the hand-made measures : cardW, cardH, cornerXmin, cornerXmax, cornerYmin and cornerYmax


In [ ]:
# Run a few times...


img_fn=random.choice(imgs_fns)
display_img(cv2.imread(img_fn,cv2.IMREAD_UNCHANGED),polygons=[refCornerHL,refCornerLR])

# Finding the convex hulls
This function 'find_hull' finds the convex hull in one of the corner of a card image.

In [ ]:
def findHull(img, corner=refCornerHL, debug="no"):
    kernel = np.ones((3,3), np.uint8)
    corner = corner.astype(int)

    # 1. Define the zone (ROI)
    x1, y1 = int(corner[0][0]), int(corner[0][1])
    x2, y2 = int(corner[2][0]), int(corner[2][1])
    w, h = x2-x1, y2-y1
    zone = img[y1:y2, x1:x2].copy()

    # 2. Color Filtering: Remove Blue (for cards with blue elements/logos)
    hsv_zone = cv2.cvtColor(zone, cv2.COLOR_BGR2HSV)
    lower_blue = np.array([100, 50, 50])
    upper_blue = np.array([130, 255, 255])
    blue_mask = cv2.inRange(hsv_zone, lower_blue, upper_blue)

    # 3. Pre-processing: Use Adaptive Thresholding (more robust than Canny for '7')
    gray = cv2.cvtColor(zone, cv2.COLOR_BGR2GRAY)
    # Use GaussianBlur to reduce noise before thresholding
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    # Adaptive threshold helps detect thin lines in the '7' and the Club stem
    thld = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                 cv2.THRESH_BINARY_INV, 11, 2)
    
    # 4. Remove Blue elements from the thresholded map
    thld = cv2.bitwise_and(thld, thld, mask=cv2.bitwise_not(blue_mask))

    # 5. Dilation: Crucial to merge '1' and '0' and parts of the '7'
    # iterations=3 ensures the Rank and Suit symbols are bridged together
    thld = cv2.dilate(thld, kernel, iterations=3)

    if debug != "no": cv2.imshow("Thresholded Zone", thld)
    
    # 6. Find Contours
    contours, _ = cv2.findContours(thld.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    min_area = 15       # Lowered to capture thin '7' segments
    min_solidity = 0.1
    concat_contour = None
    
    for c in contours:
        c = c.astype(np.int32) # NumPy 2.0 fix
        area = cv2.contourArea(c)
        if area < min_area: continue

        M = cv2.moments(c)
        if M['m00'] == 0: continue
        cx, cy = int(M['m10']/M['m00']), int(M['m01']/M['m00'])

        # --- RELAXED TOLERANCES FOR 7 AND 10 ---
        # w*0.48: Allows '1' to be captured on the far left
        # h*0.48: Allows '7' to be captured at the very top of the zone
        if abs(w/2 - cx) < w*0.48 and abs(h/2 - cy) < h*0.48:
            if concat_contour is None:
                concat_contour = c
            else:
                concat_contour = np.concatenate((concat_contour, c))

    if concat_contour is not None:
        hull = cv2.convexHull(concat_contour)
        hull_area = cv2.contourArea(hull)
        
        # --- ADJUSTED HULL LIMITS ---
        # 600: Lowered significantly to allow '7' of Clubs (which has less density)
        # 3000: Raised for '10'
        min_hull_area, max_hull_area = 600, 3000 
        
        if debug != "no":
            print(f"Hull Area: {hull_area}")

        if min_hull_area < hull_area < max_hull_area:
            return hull + corner[0]
            
    return None

In [ ]:
# Test find_hull on a random card image
# debug = "no" or "pause_always" or "pause_on_pb"
# If debug!="no", you may have to press a key to continue execution after pause
debug="no" 
img_fn=random.choice(imgs_fns)
print(img_fn)
img=cv2.imread(img_fn,cv2.IMREAD_UNCHANGED)

hullHL=findHull(img,refCornerHL,debug=debug)
hullLR=findHull(img,refCornerLR,debug=debug)
display_img(img,[refCornerHL,refCornerLR,hullHL,hullLR])

if debug!="no": cv2.destroyAllWindows()

### Load all card image, calculate their convex hulls and save the whole in a pickle file (1x)

The next times, we will directly load the pickle file 
The structure saved in the pickle file is a dictionnary named 'cards' of lists of triplets (img,hullHL,hullLR). The keys of the dictionnary are the card names ("Ad","10h",... so 52 entries in the dictionnary). 

In [ ]:
cards={}
print(imgs_dir)
#files = [f for f in os.listdir(imgs_dir) if os.path.isfile(os.path.join(imgs_dir, f))]

for suit in card_suits:
    for value in card_values:
        card_name=suit+value
        cards[card_name]=[]
        img=cv2.imread(os.path.join(imgs_dir,card_name+'.png'),cv2.IMREAD_UNCHANGED)
        hullHL=findHull(img,refCornerHL,debug="no") 
        if hullHL is None: 
            print(f"File {f} not used.")
            continue
        hullLR=findHull(img,refCornerLR,debug="no") 
        if hullLR is None: 
            print(f"File {f} not used.")
            continue
        # We store the image in "rgb" format (we don't need opencv anymore)
        img=cv2.cvtColor(img,cv2.COLOR_BGRA2RGBA)
        cards[card_name].append((img,hullHL,hullLR))
        print(f"Nb images for {card_name} : {len(cards[card_name])}")



print("Saved in :",cards_pck_fn)
pickle.dump(cards,open(cards_pck_fn,'wb'))

cv2.destroyAllWindows()

### Load the cards pickle file in 'cards'
'cards' is an instance of the class Cards
To get a random background image, call the method : cards.get_random() or cards.get_random(card_name) if you want a random card of a given value. Ex: cards.get_random('Ah')

In [ ]:
class Cards():
    def __init__(self,cards_pck_fn=cards_pck_fn):
        self._cards=pickle.load(open(cards_pck_fn,'rb'))
        # self._cards is a dictionary where keys are card names (ex:'Kc') and values are lists of (img,hullHL,hullLR) 
        self._nb_cards_by_value={k:len(self._cards[k]) for k in self._cards}
        print("Nb of cards loaded per name :", self._nb_cards_by_value)
        
    def get_random(self, card_name=None, display=False):
        if card_name is None:
            card_name= random.choice(list(self._cards.keys()))
        card,hull1,hull2=self._cards[card_name][random.randint(0,self._nb_cards_by_value[card_name]-1)]
        if display:
            if display: display_img(card,[hull1,hull2],"rgb")
        return card,card_name,hull1,hull2
    
cards = Cards()


In [ ]:
# Test: display a random card
_=cards.get_random(display=True)
# Display a random Ace of spades
#_=cards.get_random("As",display=True)

# Generating a scene
We can now generate a scene (= image of the dataset).

### To save bounding boxes annotations in Pascal VOC format 
http://host.robots.ox.ac.uk/pascal/VOC/voc2008/htmldoc/

In [ ]:
xml_body_1="""<annotation>
        <folder>FOLDER</folder>
        <filename>{FILENAME}</filename>
        <path>{PATH}</path>
        <source>
                <database>Unknown</database>
        </source>
        <size>
                <width>{WIDTH}</width>
                <height>{HEIGHT}</height>
                <depth>3</depth>
        </size>
"""
xml_object=""" <object>
                <name>{CLASS}</name>
                <pose>Unspecified</pose>
                <truncated>0</truncated>
                <difficult>0</difficult>
                <bndbox>
                        <xmin>{XMIN}</xmin>
                        <ymin>{YMIN}</ymin>
                        <xmax>{XMAX}</xmax>
                        <ymax>{YMAX}</ymax>
                </bndbox>
        </object>
"""
xml_body_2="""</annotation>        
"""

def create_voc_xml(xml_file, img_file,listbba,display=False):
    with open(xml_file,"w") as f:
        f.write(xml_body_1.format(**{'FILENAME':os.path.basename(img_file), 'PATH':img_file,'WIDTH':imgW,'HEIGHT':imgH}))
        for bba in listbba:            
            f.write(xml_object.format(**{'CLASS':bba.classname,'XMIN':bba.x1,'YMIN':bba.y1,'XMAX':bba.x2,'YMAX':bba.y2}))
        f.write(xml_body_2)
        if display: print("New xml",xml_file)
        


In [ ]:
# The original image of a card has the shape (cardH,cardW,4)
# We first paste it in a zero image of shape (imgH,imgW,4) at position decalX, decalY
# so that the original image is centerd in the zero image
decalX=int((imgW-cardW)/2)
decalY=int((imgH-cardH)/2)

def kps_to_polygon(kps):
    """
        Convert imgaug keypoints to shapely polygon
    """
    pts=[(kp.x,kp.y) for kp in kps]
    return Polygon(pts)

def hull_to_kps(hull, decalX=decalX, decalY=decalY):
    """
        Convert hull to imgaug keypoints
    """
    # hull is a cv2.Contour, shape : Nx1x2
    kps=[ia.Keypoint(x=p[0]+decalX,y=p[1]+decalY) for p in hull.reshape(-1,2)]
    kps=ia.KeypointsOnImage(kps, shape=(imgH,imgW,3))
    return kps

def kps_to_BB(kps):
    """
        Determine imgaug bounding box from imgaug keypoints
    """
    extend=3 # To make the bounding box a little bit bigger
    kpsx=[kp.x for kp in kps.keypoints]
    minx=max(0,int(min(kpsx)-extend))
    maxx=min(imgW,int(max(kpsx)+extend))
    kpsy=[kp.y for kp in kps.keypoints]
    miny=max(0,int(min(kpsy)-extend))
    maxy=min(imgH,int(max(kpsy)+extend))
    if minx==maxx or miny==maxy:
        return None
    else:
        return ia.BoundingBox(x1=minx,y1=miny,x2=maxx,y2=maxy)


# imgaug keypoints of the bounding box of a whole card
cardKP = ia.KeypointsOnImage([
    ia.Keypoint(x=decalX,y=decalY),
    ia.Keypoint(x=decalX+cardW,y=decalY),   
    ia.Keypoint(x=decalX+cardW,y=decalY+cardH),
    ia.Keypoint(x=decalX,y=decalY+cardH)
    ], shape=(imgH,imgW,3))

# imgaug transformation for the background
scaleBg=iaa.Scale({"height": imgH, "width": imgW})

def augment(img, list_kps, seq, restart=True):
    """
        Apply augmentation 'seq' to image 'img' and keypoints 'list_kps'
        If restart is False, the augmentation has been made deterministic outside the function (used for 3 cards scenario)
    """ 
    # Make sequence deterministic
    while True:
        if restart:
            myseq=seq.to_deterministic()
        else:
            myseq=seq
        # Augment image, keypoints and bbs 
        img_aug = myseq.augment_images([img])[0]
        list_kps_aug = [myseq.augment_keypoints([kp])[0] for kp in list_kps]
        list_bbs = [kps_to_BB(list_kps_aug[1]),kps_to_BB(list_kps_aug[2])]
        valid=True
        # Check the card bounding box stays inside the image
        for bb in list_bbs:
            if bb is None or int(round(bb.x2)) >= imgW or int(round(bb.y2)) >= imgH or int(bb.x1)<=0 or int(bb.y1)<=0:
                valid=False
                break
        if valid: break
        elif not restart:
            img_aug=None
            break
                
    return img_aug,list_kps_aug,list_bbs

class BBA:  # Bounding box + annotations
    def __init__(self,bb,classname):      
        self.x1=int(round(bb.x1))
        self.y1=int(round(bb.y1))
        self.x2=int(round(bb.x2))
        self.y2=int(round(bb.y2))
        self.classname=classname
    

In [ ]:
def convert_to_yolo_label(img_w, img_h, bba_list, class_mapping):
    yolo_lines = []
    for bba in bba_list:
        # Get Class ID
        class_id = class_mapping[bba.classname]
        
        # Calculate Center X, Center Y, Width, Height in pixels
        width_px = bba.x2 - bba.x1
        height_px = bba.y2 - bba.y1
        center_x_px = bba.x1 + (width_px / 2)
        center_y_px = bba.y1 + (height_px / 2)
        
        # Normalize (0.0 to 1.0)
        x = center_x_px / img_w
        y = center_y_px / img_h
        w = width_px / img_w
        h = height_px / img_h
        
        # YOLO format: <class_id> <x> <y> <w> <h>
        yolo_lines.append(f"{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
    
    return "\n".join(yolo_lines)

In [ ]:
class SceneResult:
    def __init__(self, res_img, bbas):
        self.final = res_img
        self.listbba = bbas
        
    def display(self):
        fig, ax = plt.subplots(1, figsize=(8, 8))
        ax.imshow(self.final)
        for bb in self.listbba:
            rect = patches.Rectangle((bb.x1, bb.y1), bb.x2-bb.x1, bb.y2-bb.y1,
                                     linewidth=1, edgecolor='b', facecolor='none')
            ax.add_patch(rect)
            
    def save_yolo(self, img_path, txt_path, class_mapping):
        # 1. Save Image
        plt.imsave(img_path, self.final)
        
        # 2. Convert and Save Labels
        img_h, img_w = self.final.shape[:2]
        yolo_lines = []
        for bba in self.listbba:
            class_id = class_mapping[bba.classname]
            # Normalize coordinates (YOLO format: class x_center y_center width height)
            w_px = bba.x2 - bba.x1
            h_px = bba.y2 - bba.y1
            x_center = (bba.x1 + w_px/2) / img_w
            y_center = (bba.y1 + h_px/2) / img_h
            w = w_px / img_w
            h = h_px / img_h
            yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")
        
        with open(txt_path, "w") as f:
            f.write("\n".join(yolo_lines))

In [ ]:
def generate_random_scene(card_number, random_seed=None, zoom=0.8, 
                          rotation_min=-180, rotation_max=180,
                          horizontal_projection_min=0.0, horizontal_projection_max=0.2):
    
    # Helper: Safely create a valid polygon even if coordinates "fold" or cross
    def create_valid_poly(kps):
        try:
            pts = [(kp.x, kp.y) for kp in kps]
            poly = Polygon(pts)
            if not poly.is_valid:
                # The .buffer(0) trick resolves most self-intersections
                poly = poly.buffer(0)
            return poly
        except:
            return None
    
    # Seed all random generators for reproducibility
    if random_seed is not None:
        random.seed(random_seed)
        np.random.seed(random_seed)
        ia.seed(random_seed)

    # Force dimensions to integers (NumPy 2.0 safety)
    iH, iW = int(imgH), int(imgW) 
    
    # Define the visible image area as a Polygon for border-clipping checks
    view_port = Polygon([(0, 0), (iW, 0), (iW, iH), (0, iH)])
    
    bg_img = backgrounds.get_random()
    bg_aug = scaleBg.augment_image(bg_img)
    final_img = bg_aug.copy()
    active_corners = []

    def get_transform():
        return iaa.Sequential([
            iaa.Affine(scale=zoom),
            iaa.Affine(rotate=(rotation_min, rotation_max)),
            iaa.PerspectiveTransform(scale=(horizontal_projection_min, horizontal_projection_max)),
            iaa.Affine(translate_percent={"x":(-0.25, 0.25), "y":(-0.25, 0.25)})
        ])

    for c_idx in range(card_number):
        card_img, card_val, hulla, hullb = cards.get_random()
        this_cH, this_cW = card_img.shape[:2]
        decalX, decalY = int((iW - this_cW) // 2), int((iH - this_cH) // 2)
        
        # Transparent canvas with neutral background (white) to avoid dark bleeding
        card_canvas = np.zeros((iH, iW, 4), dtype=np.uint8)
        card_canvas[:,:,0:3] = 255 
        card_canvas[:,:,3] = 0
        card_canvas[decalY:decalY+this_cH, decalX:decalX+this_cW, :] = card_img
        
        kpsa, kpsb = hull_to_kps(hulla, decalX, decalY), hull_to_kps(hullb, decalX, decalY)
        
        attempts = 0
        while attempts < 20:
            attempts += 1
            seq = get_transform().to_deterministic()
            img_aug = seq.augment_images([card_canvas])[0]
            list_kps_aug = [seq.augment_keypoints([kp])[0] for kp in [cardKP, kpsa, kpsb]]
            
            bb_a, bb_b = kps_to_BB(list_kps_aug[1]), kps_to_BB(list_kps_aug[2])
            if bb_a is None or bb_b is None: continue
            
            # --- GEOMETRY VALIDATION ---
            # Create polygons for the card body and the symbols (hulls)
            main_poly = create_valid_poly(list_kps_aug[0].keypoints[0:4])
            poly_a = create_valid_poly(list_kps_aug[1].keypoints[:])
            poly_b = create_valid_poly(list_kps_aug[2].keypoints[:])
            
            if main_poly is None or poly_a is None or poly_b is None:
                continue

            try:
                # --- BORDER VISIBILITY CHECK ---
                visibility_threshold = 0.90
                valid_new_corners = []
                
                # We only keep labels for symbols that are at least 90% visible
                if poly_a.intersection(view_port).area / poly_a.area > visibility_threshold:
                    valid_new_corners.append((poly_a, BBA(bb_a, card_val)))
                
                if poly_b.intersection(view_port).area / poly_b.area > visibility_threshold:
                    valid_new_corners.append((poly_b, BBA(bb_b, card_val)))

                # --- OCCLUSION CHECK ---
                invalid_placement = False
                temp_active_corners = []
                intersect_ratio = 0.15 
                
                for corner_poly, bba_obj in active_corners:
                    intersection = main_poly.intersection(corner_poly)
                    area_ratio = intersection.area / corner_poly.area
                    
                    if area_ratio > (1 - intersect_ratio):
                        pass # Fully hidden by new card: discard label
                    elif area_ratio > intersect_ratio:
                        invalid_placement = True # Partial "messy" cut: retry placement
                        break
                    else:
                        temp_active_corners.append((corner_poly, bba_obj)) # Safe

                if invalid_placement: continue
                
                # --- APPLY CARD TO SCENE ---
                # Add the valid corners of the new card to the list
                temp_active_corners.extend(valid_new_corners)
                
                # --- SMOOTH ALPHA BLENDING (Removes black borders) ---
                alpha = img_aug[:, :, 3].astype(float) / 255.0
                alpha_3 = np.stack([alpha] * 3, axis=-1)
                
                foreground = img_aug[:, :, 0:3].astype(float)
                background = final_img.astype(float)
                
                # Blend formula: (FG * Alpha) + (BG * (1 - Alpha))
                blended = (foreground * alpha_3) + (background * (1.0 - alpha_3))
                final_img = blended.astype(np.uint8)

                active_corners = temp_active_corners
                break # Success!
                
            except Exception:
                # If a TopologyException/GEOS error occurs, discard this attempt and retry
                continue
            
    final_bba = [item[1] for item in active_corners]
    return SceneResult(final_img, final_bba)

In [ ]:
new_scene = generate_random_scene(
    card_number=5,
    zoom=0.3, 
    rotation_min=-180, 
    rotation_max=180,
    horizontal_projection_min=0.2,
    horizontal_projection_max=0.2
)
new_scene.display()

## Generate the datasets
Typically, you want to generate a training dataset and a validation dataset of different size and different destination directory.
Modify the variable 'nb_cards_to_generate' and 'save_dir' accordingly


In [ ]:
all_classes = sorted(list(cards._cards.keys()))
class_to_id = {cls: i for i, cls in enumerate(all_classes)}

print(f"Total classes: {len(all_classes)}")

In [ ]:
def generate_yolo_dataset(target_root, num_images=100, split=0.8, base_seed=42):
    # 1. Create Directory Structure
    for sub in ['images/train', 'images/val', 'labels/train', 'labels/val']:
        os.makedirs(os.path.join(target_root, sub), exist_ok=True)
    
    print(f"Creating YOLO dataset at: {target_root}")
    
    # 2. Generate Images
    random.seed(base_seed)
    
    for i in tqdm(range(num_images)):
        # Decide if this image belongs to 'train' or 'val'
        is_train = (i < num_images * split)
        subset = "train" if is_train else "val"
        
        # Generate the scene using your previously built function
        current_seed = base_seed + i
        scene = generate_random_scene(
            card_number=random.randint(1, 5),
            random_seed=current_seed,
            zoom=random.uniform(0.2, 0.9),
            rotation_min=-180, rotation_max=180,
            horizontal_projection_min=0.0, horizontal_projection_max=0.2
        )
        
        # Define paths
        base_name = f"card_scene_{i:05d}"
        img_p = os.path.join(target_root, f"images/{subset}/{base_name}.jpg")
        txt_p = os.path.join(target_root, f"labels/{subset}/{base_name}.txt")
        
        # Save files
        scene.save_yolo(img_p, txt_p, class_to_id)

    # 3. Create data.yaml
    yaml_path = os.path.join(target_root, "data.yaml")
    # Use absolute path for the 'path' variable in YAML to avoid issues with YOLO training
    abs_root = os.path.abspath(target_root)
    
    yaml_content = [
        f"path: {abs_root}",
        "train: images/train",
        "val: images/val",
        "",
        "names:"
    ]
    for idx, name in enumerate(all_classes):
        yaml_content.append(f"  {idx}: {name}")
        
    with open(yaml_path, "w") as f:
        f.write("\n".join(yaml_content))
    
    print(f"Finished! data.yaml created at {yaml_path}")

In [ ]:
# Generate 100 images with seed 1234
generate_yolo_dataset("datasetssssss/synthetic_dataset_100", num_images=100000, base_seed=1234)